# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

In [9]:
import pandas as pd
import plotly.express as px
import numpy as np
from pathlib import Path

In [10]:
base_dir = Path.cwd()
data_path = (base_dir/'data'/'global_energy_mix.csv')
a = pd.read_csv(data_path)

source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
a['Source_Type'] = a['Source'].map(source_category)

print(f"Loaded: {len(a)} rows")
print(a.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


---
## Task 1 — Treemap: Fossil Fuel Dependency by Country

In [11]:
a_fossil = a.loc[a['Source_Type'] == 'Fossil'].copy()


fossil_colors = {
    'Coal':        '#E69F00',  
    'Oil':         '#D55E00',   
    'Natural Gas': '#56B4E9',   
}

fig1 = px.treemap(
    a_fossil,
    path=['Region', 'Country', 'Source'],  
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    title='Asia-Pacific Dominates Global Fossil Fuel Consumption — Led by China',
    custom_data=['TWh'],
)

fig1.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textinfo='label+value',
    marker=dict(
        pad=dict(t=25, l=4, r=4, b=4)
    ),
    hovertemplate='<b>%{label}</b><br>TWh: %{value:,.0f}<extra></extra>',
)

fig1.update_traces(
    root_color='lightgrey',
)

regions   = a_fossil['Region'].unique().tolist()
countries = a_fossil['Country'].unique().tolist()

grey_map = {node: '#BFBFBF' for node in regions + countries}
full_color_map = {**grey_map, **fossil_colors}

fig1 = px.treemap(
    a_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=full_color_map,
    title='Asia-Pacific Dominates Global Fossil Fuel Consumption — Led by China',
)

fig1.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textinfo='label+value',
    root_color='#E8E8E8',
    hovertemplate='<b>%{label}</b><br>TWh: %{value:,.0f}<extra></extra>',
    marker=dict(pad=dict(t=25, l=4, r=4, b=4)),
)

fig1.update_layout(
    margin=dict(t=60, l=10, r=10, b=10),
    font=dict(family='Arial', size=12),
    title_font_size=15,
)

fig1.show()

---
## Task 2 — Sunburst: Tipping Behaviour by Day and Meal Time

In [12]:
tips = px.data.tips()

tips_agg = (
    tips
    .groupby(['day', 'time', 'smoker'], as_index=False)['total_bill']
    .sum()
)

smoker_colors = {
    'No':  '#0072B2',   
    'Yes': '#E69F00',   
}

days  = tips_agg['day'].unique().tolist()
times = tips_agg['time'].unique().tolist()
grey_sunburst = {node: '#C8C8C8' for node in days + times}
full_sunburst_colors = {**grey_sunburst, **smoker_colors}

fig2 = px.sunburst(
    tips_agg,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=full_sunburst_colors,
    title='Saturday Dinner Drives the Most Spending — Non-Smokers Lead in Every Segment',
)

fig2.update_traces(
    textinfo='label+percent parent',   
    hovertemplate='<b>%{label}</b><br>Total Bill: $%{value:,.2f}<br>% of parent: %{percentParent:.1%}<extra></extra>',
    insidetextorientation='radial',
    root_color='#F0F0F0',
)

fig2.update_layout(
    margin=dict(t=60, l=10, r=10, b=10),
    font=dict(family='Arial', size=12),
    title_font_size=15,
    legend_title_text='Smoker',
)

fig2.show()

---
## Task 3 — Treemap vs Bar: Low-Carbon Energy by Country

In [13]:
a_lowcarbon = (
    a.loc[a['Source_Type'] == 'Low-carbon']
    .groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values('TWh', ascending=False)
)

a_lowcarbon['All'] = 'Low-carbon'

fig3a = px.treemap(
    a_lowcarbon,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-Carbon Energy by Country (Treemap)',
)

fig3a.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textinfo='label+value',
    root_color='#E8E8E8',
    hovertemplate='<b>%{label}</b><br>TWh: %{value:,.0f}<extra></extra>',
    marker=dict(pad=dict(t=25, l=4, r=4, b=4)),
)

fig3a.update_layout(
    margin=dict(t=60, l=10, r=10, b=10),
    coloraxis_showscale=False,
    font=dict(family='Arial', size=12),
    title_font_size=14,
)

fig3a.show()

df_bar = a_lowcarbon.sort_values('TWh', ascending=True)

fig3b = px.bar(
    df_bar,
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    color_continuous_scale='Blues',
    text='TWh',
    title='China Leads Global Low-Carbon Energy Production by a Wide Margin',
)

fig3b.update_traces(
    texttemplate='%{x:,.0f}',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>TWh: %{x:,.0f}<extra></extra>',
)

fig3b.update_layout(
    margin=dict(t=60, l=120, r=80, b=40),
    xaxis_title='TWh',
    yaxis_title='',
    coloraxis_showscale=False,
    font=dict(family='Arial', size=12),
    title_font_size=15,
    height=max(400, len(df_bar) * 28),  
)

fig3b.show()